# Overall analysis

In [ ]:
import pandas as pd

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# 0: 'mm' ;  1: 'nn' ;  2: 'xx' ;  3: 'yy' ;  4: 'ss' ;  5: 'bb' ;  6: 'rr' ;  7: 'ff' ;  8: 'xcut' ;
# 9: 'ycut' ;  10: 'nop' ;  11: 'ddr' ;  12: 'noc' ;  13: 'mac' ;  14: 'ul3' ;  15: 'tops'

# 0: 'cost_overall' ;  1: 'energy' ;  2: 'cycle' ;  3: 'edp' ;  4: 'cost' ;  5: 'idx' ;  6: 'ubuf' ;  7: 'buf' ;  8: 'bus' ;  
# 9: 'mac.1' ;  10: 'NoC' ;  11: 'NoP' ;  12: 'DRAM' ;  13: 'compute_die_area' ;  14: 'IO_die_area' ;  15: 'os_area' ;  
# 16: 'cost_compute' ;  17: 'cost_IO' ;  18: 'cost_os_overall' ;  19: 'cost_soc'

In [ ]:
data_Gemini_o = pd.read_csv('./dse_log/resnet50_72tops_Eyeriss/result.csv')
data_Gemini = pd.read_csv('./dse_log/resnet50_72tops_Eyeriss_32nm/result.csv')
data_TL = pd.read_csv('./dse_log/resnet50_72tops_Eyeriss_32nm_TL/result.csv')

In [ ]:
mapping_o2m = np.zeros(data_Gemini_o.values.shape[0])
for data_idx in range(len(mapping_o2m)):
    mapping_rst = data_Gemini_o.values[data_idx,:16] == data_Gemini.values[:,:16]
    mapping_rst = np.sum(mapping_rst, axis=1) == 16
    try:
        mapping_o2m[data_idx] = np.where(mapping_rst)[0][0]
    except:
        mapping_o2m[data_idx] = -1

In [ ]:
results_index = data_Gemini.columns[16:]
target = results_index[12]

use_idx = mapping_o2m >= 0
data_x = ((data_Gemini_o[target]/data_Gemini_o['energy'])[use_idx]).values
data_y = ((data_Gemini[target]/data_Gemini['energy'])[mapping_o2m[use_idx]]).values

axis_min = min(np.min(data_x),np.min(data_y))
axis_max = max(np.max(data_x),np.max(data_y))
axis_lim = [ axis_min-0.05*(axis_max-axis_min) , axis_max+0.05*(axis_max-axis_min) ]

ubuf_target_list = np.array([256, 512, 1024, 2048, 4096])
mac_target_list = np.array([128, 256, 512, 1024, 2048, 4096])
# for ubuf_target in ubuf_target_list:
for mac_target in mac_target_list:
    # print(data_Gemini['ul3'])
    target_idx = (data_Gemini_o['mac'])[use_idx] == mac_target # ubuf_target
    data_x_use = data_x[target_idx]
    data_y_use = data_y[target_idx]
    plt.scatter(data_x_use, data_y_use, label=str(mac_target))


# plt.scatter(data_x, data_y)
# plt.plot(axis_lim, axis_lim, '--', color='black')
plt.legend()
plt.title(target)
plt.xlabel('E ratio Gemini (12nm)')
plt.ylabel('E ratio Gemini (32nm)')

Gimini to TL

In [ ]:
results_index = data_Gemini.columns[16:]
target = results_index[9]

data_x = np.log2(data_TL[target])
data_y = np.log2(data_Gemini[target])

axis_min = min(np.min(data_x),np.min(data_y))
axis_max = max(np.max(data_x),np.max(data_y))
axis_lim = [ axis_min-0.05*(axis_max-axis_min) , axis_max+0.05*(axis_max-axis_min) ]

ubuf_target_list = np.array([256, 512, 1024, 2048, 4096])
for ubuf_target in ubuf_target_list:
    # print(data_Gemini['ul3'])
    target_idx = data_Gemini['ul3'] == ubuf_target
    # print(target_idx)
    data_x_use = data_x[target_idx]
    data_y_use = data_y[target_idx]
    plt.scatter(data_x_use, data_y_use, label=str(ubuf_target))


# plt.scatter(data_x, data_y)
plt.plot(axis_lim, axis_lim, '--', color='black')
plt.legend()
plt.title(target)
plt.xlabel('Simulation w/ Predictor')
plt.ylabel('Simulation w/o Predictor (Gemini)')

In [ ]:
results_index = data_Gemini.columns[16:]
target = results_index[3]
# plt.scatter(data_Gemini[target], data_TL[target])
# plt.title(target)
plt.hist(data_TL[target] / data_Gemini[target], bins=50)
plt.xlabel("Ratio Differate from Predictor In-the-Loop")
plt.title("Simulation on {:}".format(target))

# analyze seperate log

In [5]:
# from pyscripts.log_parser import *

import re
import pandas as pd

import numpy as np
import matplotlib.pyplot as plt

In [7]:
def parse_chiplet_log(file_path):
    
    with open(file_path, "r") as file:
        log_lines = file.readlines()

    top_data_all = []

    layer_data_preSPM = []
    sub_layer_data_preSPM = []

    layer_data_postSPM = []
    sub_layer_data_postSPM = []

    Top_pattern = re.compile(
        r'T\s+\d+/\d+\s+E:([\deE+.-]+),\s+T:(\d+),\s+Cost:([\deE+.-]+)\s+'
        r'Ubuf/Buf/Bus/Mac:([\deE+.-]+)/([\deE+.-]+)/([\deE+.-]+)/([\deE+.-]+)\s+'
        r'NoC energy = ([\deE+.-]+)\s+NoP energy = ([\deE+.-]+)\s+DRAM energy = ([\deE+.-]+)\s+'
        r'NoC\(NoC_tot_hops= (\d+)\s+NoP_tot_hops= (\d+),core_hops=(\d+), DRAM acc=(\d+)\)\s+'
        r'NoCtime = (\d+)\s+NocBW = (\d+)'
    )

    LG_pattern = re.compile(
        r'S\s+\d+/\d+\s+E:([\deE+.-]+),\s+T:(\d+),\s+Cost:([\deE+.-]+)\s+'
        r'Ubuf/Buf/Bus/Mac:([\deE+.-]+)/([\deE+.-]+)/([\deE+.-]+)/([\deE+.-]+)\s+'
        r'NoC energy = ([\deE+.-]+)\s+NoP energy = ([\deE+.-]+)\s+DRAM energy = ([\deE+.-]+)\s+'
        r'NoC\(NoC_tot_hops= (\d+)\s+NoP_tot_hops= (\d+),core_hops=(\d+), DRAM acc=(\d+)\)\s+'
        r'NoCtime = (\d+)\s+NocBW = (\d+)'
    )
    sub_LG_pattern = re.compile(
        r'\s*(\w[\w\d_]+)\s+1\s+\(([^)]+)\)\s+util:([\d.]+)/([\d.]+)\s+E:([\deE+.+-]+),\s+T:(\d+),\s+Cost:([\deE+.+-]+)\s+'
        r'Ubuf/Buf/Bus/Mac:([\deE+.+-]+)/([\deE+.+-]+)/([\deE+.+-]+)/([\deE+.+-]+).*?'
        r'NoC energy = ([\deE+.+-]+).*?DRAM energy = ([\deE+.+-]+).*?NoC\(NoC_tot_hops= (\d+)[^,]*,.*?DRAM acc=(\d+).*?'
        r'Computing_time = (\d+)'
    )

    counter = 0
    postSPM = False
    for line in log_lines:
        match_Top    = Top_pattern.search(line)
        match_LG     = LG_pattern.search(line)
        match_sub_LG = sub_LG_pattern.search(line)

        ## check if is before SPM or not
        line = line.strip()
        if line.startswith("Schtree_spm (after SPM search):"):
            counter = 0
            postSPM = True

        if match_Top:
            Top_data = {
                "energy":       float(match_Top.group(1)),
                "latency":      int(  match_Top.group(2)),
                "cost":         float(match_Top.group(3)),
                "ubuf":         float(match_Top.group(4)),
                "buf":          float(match_Top.group(5)),
                "bus":          float(match_Top.group(6)),
                "mac":          float(match_Top.group(7)),
                "NoC_energy":   float(match_Top.group(8)),
                "NoP_energy":   float(match_Top.group(9)),
                "DRAM_energy":  float(match_Top.group(10)),
                "NoC_tot_hops": int(  match_Top.group(11)),
                "NoP_tot_hops": int(  match_Top.group(12)),
                "core_hops":    int(  match_Top.group(13)),
                "DRAM_acc":     int(  match_Top.group(14)),
                "NoCtime":      int(  match_Top.group(15)),
                "NoC_BW":       int(  match_Top.group(16)),
                "LG_iter":      0
            }
            top_data_all.append(Top_data)

        elif match_LG:
            counter = counter+1
            LG_data = {
                "energy":       float(match_LG.group(1)),
                "latency":      int(  match_LG.group(2)),
                "cost":         float(match_LG.group(3)),
                "ubuf":         float(match_LG.group(4)),
                "buf":          float(match_LG.group(5)),
                "bus":          float(match_LG.group(6)),
                "mac":          float(match_LG.group(7)),
                "NoC_energy":   float(match_LG.group(8)),
                "NoP_energy":   float(match_LG.group(9)),
                "DRAM_energy":  float(match_LG.group(10)),
                "NoC_tot_hops": int(  match_LG.group(11)),
                "NoP_tot_hops": int(  match_LG.group(12)),
                "core_hops":    int(  match_LG.group(13)),
                "DRAM_acc":     int(  match_LG.group(14)),
                "NoCtime":      int(  match_LG.group(15)),
                "NoC_BW":       int(  match_LG.group(16)),
                "LG_iter":      int(  counter)
            }
            if postSPM:
                layer_data_postSPM.append(LG_data)
            else:
                layer_data_preSPM.append(LG_data)

        elif match_sub_LG:
            sub_LG_data = {
                "layer": match_sub_LG.group(1),
                "params": match_sub_LG.group(2),
                "util_actual":      float(match_sub_LG.group(3)),
                "util_peak":        float(match_sub_LG.group(4)),
                "energy":           float(match_sub_LG.group(5)),
                "latency":          int(  match_sub_LG.group(6)),
                "cost":             float(match_sub_LG.group(7)),
                "ubuf":             float(match_sub_LG.group(8)),
                "buf":              float(match_sub_LG.group(9)),
                "bus":              float(match_sub_LG.group(10)),
                "mac":              float(match_sub_LG.group(11)),
                "NoC_energy":       float(match_sub_LG.group(12)),
                "DRAM_energy":      float(match_sub_LG.group(13)),
                "NoC_hops":         int(  match_sub_LG.group(14)),
                "DRAM_acc":         int(  match_sub_LG.group(15)),
                "Computing_time":   int(  match_sub_LG.group(16)),
                "LG_iter":          int(  counter)
            }
            if postSPM:
                sub_layer_data_postSPM.append(sub_LG_data)
            else:
                sub_layer_data_preSPM.append(sub_LG_data)

    preSPM_data = {
        'top':   top_data_all[0], 
        'LG':    pd.DataFrame(layer_data_preSPM),
        'layer': pd.DataFrame(sub_layer_data_preSPM)
    }

    postSPM_data = {
        'top':   top_data_all[1], 
        'LG':    pd.DataFrame(layer_data_postSPM),
        'layer': pd.DataFrame(sub_layer_data_postSPM)
    }

    return preSPM_data, postSPM_data

In [8]:
preSPM_data, postSPM_data = parse_chiplet_log("./dse_log/resnet50_72tops_Eyeriss_32nm_TL/resnet50/DRAM_bw_294912/NoC_bw_8/ul3_256/NoP_bw_4/mac_num_128/xcut_2_ycut_1/out.log")

In [ ]:
# plot bar chat
plt_data_pre = preSPM_data['top']
plt_data_pos = postSPM_data['top']



          energy  latency          cost         ubuf         buf       bus  \
0   2.589000e+08    16128  4.176000e+12   12870000.0  24430000.0   30440.0   
1   3.256000e+08     5037  1.640000e+12  114700000.0  27480000.0       0.0   
2   2.586000e+08    12537  3.242000e+12   12830000.0  11060000.0   45130.0   
3   4.654000e+08    18128  8.436000e+12   29650000.0  13450000.0  102900.0   
4   2.687000e+08     8543  2.296000e+12   22380000.0  13010000.0   92230.0   
5   3.256000e+08     5037  1.640000e+12  114700000.0  27480000.0       0.0   
6   4.668000e+08    18622  8.693000e+12   29650000.0  13450000.0  102900.0   
7   2.686000e+08     8543  2.295000e+12   22380000.0  13010000.0   92230.0   
8   3.256000e+08     5037  1.640000e+12  114700000.0  27480000.0       0.0   
9   4.560000e+08    21504  9.806000e+12   17560000.0  11290000.0   44370.0   
10  2.271000e+08     6003  1.363000e+12   14390000.0   6787000.0   46120.0   
11  3.399000e+08     4557  1.549000e+12  131000000.0  28300000.0